# 🛒 Shopper Spectrum: Customer Segmentation and Product Recommendations in E-Commerce

## Objectives
- Data Cleaning & Preprocessing
- Exploratory Data Analysis (EDA)
- RFM Analysis
- Customer Segmentation using KMeans
- Product Recommendation using Item-Based Collaborative Filtering
- Model Saving for Streamlit Deployment
- Business Insights


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)


In [ ]:

df = pd.read_csv('online_retail.csv', encoding='latin1')
print(df.shape)
df.head()


## Data Cleaning

In [ ]:
# Convert datetime safely
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate'],
    format='mixed',
    errors='coerce'
)

# Remove invalid dates
df.dropna(subset=['InvoiceDate'], inplace=True)

# Remove missing CustomerID
df.dropna(subset=['CustomerID'], inplace=True)

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove invalid quantity and price
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Feature Engineering
df['CustomerID'] = df['CustomerID'].astype(int)
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

print('Cleaned Shape:', df.shape)
df.head()


## EDA

In [ ]:

plt.figure(figsize=(12,5))
df['Country'].value_counts().head(10).plot(kind='bar')
plt.title('Top Countries by Transactions')
plt.show()


In [ ]:

plt.figure(figsize=(12,5))
df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10).plot(kind='bar')
plt.title('Top Selling Products')
plt.show()


In [ ]:

monthly_sales = df.set_index('InvoiceDate').resample('M')['TotalAmount'].sum()

plt.figure(figsize=(12,5))
monthly_sales.plot()
plt.title('Sales Trend')
plt.show()


## RFM Analysis

In [ ]:

snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo':'nunique',
    'TotalAmount':'sum'
})

rfm.columns = ['Recency','Frequency','Monetary']
rfm.head()


In [ ]:

sns.histplot(rfm['Recency'], kde=True)
plt.title('Recency Distribution')
plt.show()

sns.histplot(rfm['Frequency'], kde=True)
plt.title('Frequency Distribution')
plt.show()

sns.histplot(rfm['Monetary'], kde=True)
plt.title('Monetary Distribution')
plt.show()


## Scaling & Cluster Selection

In [ ]:

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

inertia = []
sil_scores = []

for k in range(2,11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)

    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels))

plt.plot(range(2,11), inertia, marker='o')
plt.title('Elbow Curve')
plt.show()

plt.plot(range(2,11), sil_scores, marker='o')
plt.title('Silhouette Score')
plt.show()


In [ ]:

best_k = 4

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

cluster_summary = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean()
cluster_summary


In [ ]:

cluster_labels = {
0:'High-Value',
1:'Regular',
2:'Occasional',
3:'At-Risk'
}

rfm['Segment'] = rfm['Cluster'].map(cluster_labels)
rfm[['Cluster','Segment']].head()


In [ ]:

plt.figure(figsize=(10,6))
sns.scatterplot(data=rfm,x='Frequency',y='Monetary',hue='Segment')
plt.title('Customer Segments')
plt.show()


## Product Recommendation System

In [ ]:

customer_product = pd.crosstab(df['CustomerID'],df['Description'])

similarity = cosine_similarity(customer_product.T)

similarity_df = pd.DataFrame(
    similarity,
    index=customer_product.columns,
    columns=customer_product.columns
)

plt.figure(figsize=(10,8))
sns.heatmap(similarity_df.iloc[:20,:20])
plt.title('Product Similarity Heatmap')
plt.show()


In [ ]:

def recommend_products(product_name, top_n=5):
    if product_name not in similarity_df.index:
        return ['Product Not Found']

    recs = similarity_df[product_name].sort_values(ascending=False)[1:top_n+1]
    return list(recs.index)

recommend_products(similarity_df.index[0])


## Save Models

In [ ]:

joblib.dump(kmeans,'kmeans_model.pkl')
joblib.dump(scaler,'scaler.pkl')
joblib.dump(similarity_df,'product_similarity.pkl')

rfm.to_csv('customer_segments.csv')



# Business Insights

### High-Value Customers
- Frequent buyers
- Highest spending customers
- Ideal for loyalty programs

### Regular Customers
- Consistent purchasing behaviour
- Target with cross-selling campaigns

### Occasional Customers
- Purchase infrequently
- Can be nurtured through promotions

### At-Risk Customers
- Long time since last purchase
- Require retention campaigns


# 🛒 Shopper Spectrum: Customer Segmentation and Product Recommendations in E-Commerce

## Project Repository
[GitHub Repository](https://github.com/Sumit-Tomar-Web-Developer/Labmentix_internship/tree/ac2cc1699712a8d018f9d5b769198e61ee415cf5/Shopper%20Spectrum%20Customer%20Segmentation%20and%20Product%20Recommendations%20in%20E-Commerce)

## Objectives
- Data Cleaning & Preprocessing
- Exploratory Data Analysis (EDA)
- RFM Analysis
- Customer Segmentation using KMeans
- Product Recommendation using Item-Based Collaborative Filtering
- Model Saving for Streamlit Deployment
- Business Insights
